In [0]:
import json

import pyspark.sql.functions as F
from pyspark.sql.window import Window

from common_utils.logging import get_logger
from common_utils.generic_functions import safe_cast

dbutils.widgets.text("path","")
config_path=dbutils.widgets.get("path")
logger=get_logger("fact_sales_gold")
logger.info("Reading configuration files from %s",config_path)

with open(config_path,"r") as f:
    config=json.load(f)
logger.info("Gold configuration loaded successfully")

# 1. Source / Target Configuration
source_config=config['source']
target_config=config['target']
source_name=source_config['source_name']
sales_orders_table=source_config["tables"]["sales_orders"]
order_products_table=source_config["tables"]["order_products"]
promotions_table=source_config["tables"]["promotions"]
target_table=target_config["target_table"]
logger.info("%s source table configured",source_name)
logger.info("[%s] Target table: %s",source_name,target_table)

# 2. Read Silver Tables
logger.info("[%s] reading sales_orders from path [%s]",source_name,sales_orders_table)
sales_orders_df=spark.read.table(sales_orders_table)
logger.info("[%s] reading order_products from path [%s]",source_name,order_products_table)
order_products_df=spark.read.table(order_products_table)
logger.info("[%s] reading promotions from path [%s]",source_name,promotions_table)
promotions_df=spark.read.table(promotions_table)
logger.info("[%s] All silver tables successfully loaded",source_name)

def prepare_promotions(df,config):
    logger.info("[%s] Preparing promotions",source_name)
    df=df.select("order_number","promo_product_id","discount","promo_quantity")
    df=safe_cast(df,{
        "order_number":"long",
        "promo_product_id":"string",
        "discount":"double",
        "promo_quantity":"long"
    })
    return df

def transform_fact_sales(sales_order_df,order_products_df,promotions_df,config):
    # 7.1 Prepare orders
    logger.info("[%s] preparing sales_orders",source_name)
    orders=(sales_orders_df.select("order_number","customer_id","order_timestamp")
            .withColumn("order_date_key",F.date_format(F.to_timestamp(F.col("order_timestamp")),config["transform"]["order_date_key_format"]).cast("int")).select("order_number","customer_id","order_timestamp","order_date_key"))

    # 7.2 Prepare order products
    logger.info("[%s] Preparing order products",source_name)
    products=(order_products_df.select("order_number","customer_id",F.col("id").alias("product_id"),F.col("qty").alias("qty"),F.col("price").alias("unit_price")))
    # 7.3 Prepare promotions 
    promotions=prepare_promotions(promotions_df,config)
    # 7.4 Deduplicate orders - keep latest record per order_number by order_timestamp
    logger.info("[%s] Deduplicating sales_orders to keep latest per order_number",source_name)
    win=Window.partitionBy(F.col("order_number")).orderBy(F.col("order_timestamp").desc())
    orders=(orders.withColumn("rank",F.row_number().over(win)).filter(F.col("rank")==1).drop("rank"))

    # 7.4b Join deduplicated orders with order products
    logger.info("[%s] joining sales_orders with order_products",source_name)
    fact=(products.alias("p").join(orders.alias("o"),F.col("p.order_number")==F.col("o.order_number"),"left"))

    #====================================================
    # 7.5 Join promotions
    #====================================================
    logger.info("[%s] joining promotions", source_name)
    fact=(fact.join(promotions.alias("pr"),(F.col("p.order_number")==F.col("pr.order_number")) & (F.col("p.product_id")==F.col("pr.promo_product_id")),"left"))
    # 7.6 Select Fact columns
    #====================================================
    logger.info("[%s] selection fact_sales columns", source_name)
    
    fact=fact.select(
                        # Key
        F.col("p.order_number").alias("order_number"),
        F.col("o.customer_id").alias("customer_id"),
        F.col("p.product_id").alias("product_id"),
        F.col("o.order_date_key").alias("order_date_key"),
                # Measures
        F.col("p.qty").alias("qty"),
        F.col("p.unit_price").alias("unit_price"),
                                    # Promotion Info
        F.coalesce(F.col("pr.discount"),F.lit(config["transform"]["default_discount"])).alias("discount"),
        F.coalesce(F.col("pr.promo_quantity"),F.lit(config["transform"]["default_promo_quantity"])).alias("promo_quantity") 
    )
    # 7.7 Derive total amount
    logger.info("[%s] Calculating total amount",source_name)
    fact=fact.withColumn("total_amount",F.round(F.col("qty")*F.col("unit_price")*(F.lit(1)-F.col("discount")),2))
    # 7.8 Cast final columns
    logger.info("[%s] Applying final data types", source_name)
    fact=safe_cast(fact,config.get("cast_columns",{}))
    # 7.9 Select final columns
    logger.info("[%s] select final columns")
    fact.select(*config["select_columns"])
    # 7.10 Remove duplicates
    logger.info("[%s] Remove Duplicates from fact records",source_name)
    fact=fact.dropDuplicates(["order_number","product_id"])
    logger.info("[%s] fact_sales transformation completed.",source_name)

    return fact

logger.info("[%s] Starting gold fact_sales creation",source_name)
fact_sales=transform_fact_sales(sales_orders_df,order_products_df,promotions_df,config)
logger.info("[%s] Validating fact_sales",source_name)
record_count=[fact_sales.count()]
logger.info("[%s] fact_sales record count: %s",source_name,record_count)

if record_count==0:
    raise ValueError("fact_sales contains zero records.")

# Check for duplicate grains
duplicate_count=(fact_sales.groupBy("order_number","product_id").count().filter(F.col("count")>1).count())
logger.info("[%s] Duplicate fact grain count: %s",source_name,duplicate_count)

if duplicate_count>0:
    raise ValueError("Duplicate order_number + product_id ""records detected in fact_sales")
# 10. Display fact
fact_sales.show()
(fact_sales.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(target_table))
